In [13]:
import os
import json

with open('celebrity_file_names_dict.json', 'r') as f:
    names_dict = json.load(f)

In [14]:
count = 0
names_scale = {}
for names in names_dict.keys():
    names_scale[names] = count
    count += 1
names_scale

{'charles leclerc': 0,
 'dhoni': 1,
 'lewis hamilton': 2,
 'lionel messi': 3,
 'max verstappen': 4,
 'neymar': 5,
 'rohit sharma': 6,
 'ronaldo': 7,
 'virat kohli': 8}

In [9]:
import numpy as np
import cv2
import pywt

def wavelet_transform(image, wavelet='haar', level=1):
    img_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    img_gray = np.float32(img_gray) / 255.0

    coeffs_h = pywt.wavedec2(img_gray, wavelet, level=level)
    
    coeffs = list(coeffs_h)
    coeffs[0] *= 0

    img_reconstructed = pywt.waverec2(coeffs, wavelet)

    img_reconstructed = np.clip(img_reconstructed, 0, 1)

    return (img_reconstructed * 255).astype(np.uint8)
    

In [17]:
X , y = [], []
for names, files in names_dict.items():
    for file in files:
        img = cv2.imread(file)
        if img is None:
            continue
        wavelet_img = wavelet_transform(img)
        scaled_img_raw = cv2.resize(img, (32, 32))
        scaled_img_wavelet = cv2.resize(wavelet_img, (32, 32))
        combined_img = np.vstack((scaled_img_raw.reshape(32*32*3, 1), scaled_img_wavelet.reshape(32*32, 1)))
        X.append(combined_img)
        y.append(names_scale[names])

In [20]:
X = np.array(X).reshape(len(X), 32*32*3 + 32*32).astype(float)